# 01 — Data Loading & Cleaning

This notebook loads the raw WITS trade dataset, inspects its schema,
runs the preprocessing pipeline (energy filtering, cleaning, ISO-3 enrichment),
and saves processed CSVs to `data/processed/`.

**Prerequisites:** Place the raw WITS file in `data/raw/` (see `data/raw/README.md`).

In [1]:
import sys
from pathlib import Path

# Ensure the project root is on the Python path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np

from src.config import DATA_RAW, DATA_PROCESSED, PROCESSED_ENERGY_TRADE
from src.data_loader import load_raw, find_raw_file
from src.preprocess import run_pipeline
from src.utils import report_country_matching

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 30)

## 1. Locate and Load Raw Data

In [2]:
raw_path = find_raw_file()
print(f"Found raw file: {raw_path.name}")
print(f"File size: {raw_path.stat().st_size / 1e6:.1f} MB")

Found raw file: comtrade_energy_trade.csv
File size: 125.1 MB


In [3]:
df_raw = load_raw(raw_path)

Loading raw data from: comtrade_energy_trade.csv
  Raw shape: (393778, 47)
  Raw columns: ['typeCode', 'freqCode', 'refPeriodId', 'refYear', 'refMonth', 'period', 'reporterCode', 'reporterISO', 'reporterDesc', 'flowCode', 'flowDesc', 'partnerCode', 'partnerISO', 'partnerDesc', 'partner2Code', 'partner2ISO', 'partner2Desc', 'classificationCode', 'classificationSearchCode', 'isOriginalClassification', 'cmdCode', 'cmdDesc', 'aggrLevel', 'isLeaf', 'customsCode', 'customsDesc', 'mosCode', 'motCode', 'motDesc', 'qtyUnitCode', 'qtyUnitAbbr', 'qty', 'isQtyEstimated', 'altQtyUnitCode', 'altQtyUnitAbbr', 'altQty', 'isAltQtyEstimated', 'netWgt', 'isNetWgtEstimated', 'grossWgt', 'isGrossWgtEstimated', 'cifvalue', 'fobvalue', 'primaryValue', 'legacyEstimationFlag', 'isReported', 'isAggregate']
  Normalized shape: (393778, 46)
  Normalized columns: ['type_code', 'freq_code', 'ref_period_id', 'year', 'ref_month', 'reporter_code', 'reporter_iso3', 'reporter', 'flow_code', 'flow', 'partner_code', 'part

## 2. Inspect Raw Data

In [4]:
print(f"Shape: {df_raw.shape}")
print(f"\nColumns: {list(df_raw.columns)}")
print(f"\nData types:")
print(df_raw.dtypes)

Shape: (393778, 46)

Columns: ['type_code', 'freq_code', 'ref_period_id', 'year', 'ref_month', 'reporter_code', 'reporter_iso3', 'reporter', 'flow_code', 'flow', 'partner_code', 'partner_iso3', 'partner', 'partner2code', 'partner2iso', 'partner2desc', 'classification_code', 'classification_search_code', 'is_original_classification', 'product_code', 'product', 'aggr_level', 'is_leaf', 'customs_code', 'customs_desc', 'mos_code', 'mot_code', 'mot_desc', 'qty_unit_code', 'qty_unit_abbr', 'qty', 'is_qty_estimated', 'alt_qty_unit_code', 'alt_qty_unit_abbr', 'alt_qty', 'is_alt_qty_estimated', 'net_wgt', 'is_net_wgt_estimated', 'gross_wgt', 'is_gross_wgt_estimated', 'cif_value', 'fob_value', 'trade_value_usd', 'legacy_estimation_flag', 'is_reported', 'is_aggregate']

Data types:
type_code                     str
freq_code                     str
ref_period_id               int64
year                        Int64
ref_month                   int64
                           ...   
fob_value     

In [5]:
df_raw.head(10)

,type_code,freq_code,ref_period_id,year,ref_month,reporter_code,reporter_iso3,reporter,flow_code,flow,partner_code,partner_iso3,partner,partner2code,partner2iso,partner2desc,classification_code,classification_search_code,is_original_classification,product_code,product,aggr_level,is_leaf,customs_code,customs_desc,mos_code,mot_code,mot_desc,qty_unit_code,qty_unit_abbr,qty,is_qty_estimated,alt_qty_unit_code,alt_qty_unit_abbr,alt_qty,is_alt_qty_estimated,net_wgt,is_net_wgt_estimated,gross_wgt,is_gross_wgt_estimated,cif_value,fob_value,trade_value_usd,legacy_estimation_flag,is_reported,is_aggregate
0,C,A,20000101,2000,52,8,ALB,Albania,M,Import,0,W00,World,0,W00,World,H1,HS,True,27,"Mineral fuels, mineral oils and products of th...",2,False,C00,TOTAL CPC,0,0,TOTAL MOT,-1,NaN,NaN,False,-1,NaN,NaN,False,NaN,False,NaN,False,98348640.0,NaN,98348640.0,0,False,False
1,C,A,20000101,2000,52,8,ALB,Albania,M,Import,40,AUT,Austria,0,W00,World,H1,HS,True,27,"Mineral fuels, mineral oils and products of th...",2,False,C00,TOTAL CPC,0,0,TOTAL MOT,-1,NaN,NaN,False,-1,NaN,NaN,False,NaN,False,NaN,False,14726.0,NaN,14726.0,0,False,False
2,C,A,20000101,2000,52,8,ALB,Albania,M,Import,56,BEL,Belgium,0,W00,World,H1,HS,True,27,"Mineral fuels, mineral oils and products of th...",2,False,C00,TOTAL CPC,0,0,TOTAL MOT,-1,NaN,NaN,False,-1,NaN,NaN,False,NaN,False,NaN,False,14923.0,NaN,14923.0,0,False,False
3,C,A,20000101,2000,52,8,ALB,Albania,M,Import,100,BGR,Bulgaria,0,W00,World,H1,HS,True,27,"Mineral fuels, mineral oils and products of th...",2,False,C00,TOTAL CPC,0,0,TOTAL MOT,-1,NaN,NaN,False,-1,NaN,NaN,False,NaN,False,NaN,False,71227.0,NaN,71227.0,0,False,False
4,C,A,20000101,2000,52,8,ALB,Albania,M,Import,112,BLR,Belarus,0,W00,World,H1,HS,True,27,"Mineral fuels, mineral oils and products of th...",2,False,C00,TOTAL CPC,0,0,TOTAL MOT,-1,NaN,NaN,False,-1,NaN,NaN,False,NaN,False,NaN,False,3024.0,NaN,3024.0,0,False,False
5,C,A,20000101,2000,52,8,ALB,Albania,M,Import,191,HRV,Croatia,0,W00,World,H1,HS,True,27,"Mineral fuels, mineral oils and products of th...",2,False,C00,TOTAL CPC,0,0,TOTAL MOT,-1,NaN,NaN,False,-1,NaN,NaN,False,NaN,False,NaN,False,2273.0,NaN,2273.0,0,False,False
6,C,A,20000101,2000,52,8,ALB,Albania,M,Import,196,CYP,Cyprus,0,W00,World,H1,HS,True,27,"Mineral fuels, mineral oils and products of th...",2,False,C00,TOTAL CPC,0,0,TOTAL MOT,-1,NaN,NaN,False,-1,NaN,NaN,False,NaN,False,NaN,False,640257.0,NaN,640257.0,0,False,False
7,C,A,20000101,2000,52,8,ALB,Albania,M,Import,203,CZE,Czechia,0,W00,World,H1,HS,True,27,"Mineral fuels, mineral oils and products of th...",2,False,C00,TOTAL CPC,0,0,TOTAL MOT,-1,NaN,NaN,False,-1,NaN,NaN,False,NaN,False,NaN,False,29.0,NaN,29.0,0,False,False
8,C,A,20000101,2000,52,8,ALB,Albania,M,Import,251,FRA,France,0,W00,World,H1,HS,True,27,"Mineral fuels, mineral oils and products of th...",2,False,C00,TOTAL CPC,0,0,TOTAL MOT,-1,NaN,NaN,False,-1,NaN,NaN,False,NaN,False,NaN,False,12348.0,NaN,12348.0,0,False,False
9,C,A,20000101,2000,52,8,ALB,Albania,M,Import,276,DEU,Germany,0,W00,World,H1,HS,True,27,"Mineral fuels, mineral oils and products of th...",2,False,C00,TOTAL CPC,0,0,TOTAL MOT,-1,NaN,NaN,False,-1,NaN,NaN,False,NaN,False,NaN,False,209663.0,NaN,209663.0,0,False,False


In [6]:
# Missing values
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(1)
pd.DataFrame({'missing': missing, 'pct': missing_pct}).sort_values('pct', ascending=False)

,missing,pct
qty_unit_abbr,393769,100.0
alt_qty_unit_abbr,393778,100.0
net_wgt,221440,56.2
gross_wgt,217799,55.3
qty,191916,48.7
...,...,...
classification_search_code,0,0.0
is_original_classification,0,0.0
product_code,0,0.0
product,0,0.0


In [7]:
# Quick value counts for key columns
for col in ['reporter', 'partner', 'year', 'product', 'product_code', 'flow']:
    if col in df_raw.columns:
        print(f"\n── {col} ({df_raw[col].nunique()} unique) ──")
        print(df_raw[col].value_counts().head(10))


── reporter (203 unique) ──
reporter
France            7060
Netherlands       6957
USA               6848
Belgium           6607
China             6528
Germany           6179
United Kingdom    5952
Spain             5904
Italy             5899
Canada            5815
Name: count, dtype: int64

── partner (251 unique) ──
partner
World             8084
USA               6686
France            6168
United Kingdom    6153
Germany           6100
China             5981
Netherlands       5826
Italy             5464
Belgium           5451
Spain             5304
Name: count, dtype: int64

── year (24 unique) ──
year
2018    18623
2023    18550
2017    18486
2019    18449
2021    18435
2022    18368
2016    18062
2020    17965
2015    17810
2014    17461
Name: count, dtype: Int64

── product (1 unique) ──
product
Mineral fuels, mineral oils and products of their distillation; bituminous substances; mineral waxes    393778
Name: count, dtype: int64

── product_code (1 unique) ──
product_code
27  

## 3. Run Preprocessing Pipeline

This filters to energy products, cleans invalid rows, adds ISO-3 codes, and builds summary datasets.

In [8]:
results = run_pipeline(df_raw)

df_energy = results['energy_trade']
df_country = results['country_summary']
df_partner = results['partner_summary']


=== Preprocessing Pipeline ===
  Energy filter: 393,778 / 393,778 rows retained (100.0%)
  Dropped 0 rows with missing required fields.
  Clean shape: (393777, 46)
  Adding ISO-3 country codes…
  Building country summary…
  Building partner summary…
  Saved → energy_trade.csv  (393,777 rows)
  Saved → country_summary.csv  (4,143 rows)
  Saved → partner_summary.csv  (279,879 rows)


## 4. Inspect Processed Data

In [9]:
print(f"Energy trade: {df_energy.shape}")
print(f"Country summary: {df_country.shape}")
print(f"Partner summary: {df_partner.shape}")

Energy trade: (393777, 48)
Country summary: (4143, 5)
Partner summary: (279879, 4)


In [10]:
df_energy.head(10)

,type_code,freq_code,ref_period_id,year,ref_month,reporter_code,reporter_iso3,reporter,flow_code,flow,partner_code,partner_iso3,partner,partner2code,partner2iso,partner2desc,classification_code,classification_search_code,is_original_classification,product_code,product,aggr_level,is_leaf,customs_code,customs_desc,mos_code,mot_code,mot_desc,qty_unit_code,qty_unit_abbr,qty,is_qty_estimated,alt_qty_unit_code,alt_qty_unit_abbr,alt_qty,is_alt_qty_estimated,net_wgt,is_net_wgt_estimated,gross_wgt,is_gross_wgt_estimated,cif_value,fob_value,trade_value_usd,legacy_estimation_flag,is_reported,is_aggregate,reporter_std,partner_std
0,C,A,20000101,2000,52,8,ALB,Albania,M,Import,0,NaN,World,0,W00,World,H1,HS,True,27,"Mineral fuels, mineral oils and products of th...",2,False,C00,TOTAL CPC,0,0,TOTAL MOT,-1,NaN,NaN,False,-1,NaN,NaN,False,NaN,False,NaN,False,98348640.0,NaN,98348640.0,0,False,False,Albania,NaN
1,C,A,20000101,2000,52,8,ALB,Albania,M,Import,40,AUT,Austria,0,W00,World,H1,HS,True,27,"Mineral fuels, mineral oils and products of th...",2,False,C00,TOTAL CPC,0,0,TOTAL MOT,-1,NaN,NaN,False,-1,NaN,NaN,False,NaN,False,NaN,False,14726.0,NaN,14726.0,0,False,False,Albania,Austria
2,C,A,20000101,2000,52,8,ALB,Albania,M,Import,56,BEL,Belgium,0,W00,World,H1,HS,True,27,"Mineral fuels, mineral oils and products of th...",2,False,C00,TOTAL CPC,0,0,TOTAL MOT,-1,NaN,NaN,False,-1,NaN,NaN,False,NaN,False,NaN,False,14923.0,NaN,14923.0,0,False,False,Albania,Belgium
3,C,A,20000101,2000,52,8,ALB,Albania,M,Import,100,BGR,Bulgaria,0,W00,World,H1,HS,True,27,"Mineral fuels, mineral oils and products of th...",2,False,C00,TOTAL CPC,0,0,TOTAL MOT,-1,NaN,NaN,False,-1,NaN,NaN,False,NaN,False,NaN,False,71227.0,NaN,71227.0,0,False,False,Albania,Bulgaria
4,C,A,20000101,2000,52,8,ALB,Albania,M,Import,112,BLR,Belarus,0,W00,World,H1,HS,True,27,"Mineral fuels, mineral oils and products of th...",2,False,C00,TOTAL CPC,0,0,TOTAL MOT,-1,NaN,NaN,False,-1,NaN,NaN,False,NaN,False,NaN,False,3024.0,NaN,3024.0,0,False,False,Albania,Belarus
5,C,A,20000101,2000,52,8,ALB,Albania,M,Import,191,HRV,Croatia,0,W00,World,H1,HS,True,27,"Mineral fuels, mineral oils and products of th...",2,False,C00,TOTAL CPC,0,0,TOTAL MOT,-1,NaN,NaN,False,-1,NaN,NaN,False,NaN,False,NaN,False,2273.0,NaN,2273.0,0,False,False,Albania,Croatia
6,C,A,20000101,2000,52,8,ALB,Albania,M,Import,196,CYP,Cyprus,0,W00,World,H1,HS,True,27,"Mineral fuels, mineral oils and products of th...",2,False,C00,TOTAL CPC,0,0,TOTAL MOT,-1,NaN,NaN,False,-1,NaN,NaN,False,NaN,False,NaN,False,640257.0,NaN,640257.0,0,False,False,Albania,Cyprus
7,C,A,20000101,2000,52,8,ALB,Albania,M,Import,203,CZE,Czechia,0,W00,World,H1,HS,True,27,"Mineral fuels, mineral oils and products of th...",2,False,C00,TOTAL CPC,0,0,TOTAL MOT,-1,NaN,NaN,False,-1,NaN,NaN,False,NaN,False,NaN,False,29.0,NaN,29.0,0,False,False,Albania,Czechia
8,C,A,20000101,2000,52,8,ALB,Albania,M,Import,251,FRA,France,0,W00,World,H1,HS,True,27,"Mineral fuels, mineral oils and products of th...",2,False,C00,TOTAL CPC,0,0,TOTAL MOT,-1,NaN,NaN,False,-1,NaN,NaN,False,NaN,False,NaN,False,12348.0,NaN,12348.0,0,False,False,Albania,France
9,C,A,20000101,2000,52,8,ALB,Albania,M,Import,276,DEU,Germany,0,W00,World,H1,HS,True,27,"Mineral fuels, mineral oils and products of th...",2,False,C00,TOTAL CPC,0,0,TOTAL MOT,-1,NaN,NaN,False,-1,NaN,NaN,False,NaN,False,NaN,False,209663.0,NaN,209663.0,0,False,False,Albania,Germany


In [11]:
df_country.head(10)

,country,year,total_imports,total_exports,trade_balance
0,Afghanistan,2008,9.751530e+08,0.00,-9.751530e+08
1,Afghanistan,2009,1.576459e+09,787416.00,-1.575672e+09
2,Afghanistan,2010,2.150375e+09,0.00,-2.150375e+09
3,Afghanistan,2011,4.444218e+09,4782560.00,-4.439435e+09
4,Afghanistan,2012,3.035052e+09,0.00,-3.035052e+09
5,Afghanistan,2013,2.905095e+09,0.00,-2.905095e+09
6,Afghanistan,2014,2.977248e+09,0.00,-2.977248e+09
7,Afghanistan,2015,3.275914e+09,39432154.00,-3.236482e+09
8,Afghanistan,2016,2.016164e+09,55345232.00,-1.960819e+09
9,Afghanistan,2017,1.847084e+09,87093219.14,-1.759990e+09


In [12]:
df_partner.head(10)

,reporter,partner,year,total_trade_value
0,Afghanistan,"Areas, nes",2012,1.399301e+09
1,Afghanistan,"Areas, nes",2013,1.067075e+09
2,Afghanistan,"Areas, nes",2014,9.849450e+08
3,Afghanistan,"Areas, nes",2017,5.325976e+07
4,Afghanistan,"Areas, nes",2018,8.644079e+07
5,Afghanistan,"Areas, nes",2019,5.271730e+07
6,Afghanistan,Aruba,2017,3.384100e+02
7,Afghanistan,Australia,2011,5.160000e+02
8,Afghanistan,Azerbaijan,2011,6.916726e+07
9,Afghanistan,Azerbaijan,2015,3.594838e+07


## 5. Country Matching Quality

In [13]:
if 'reporter_iso3' in df_energy.columns:
    summary, unmatched = report_country_matching(df_energy, col='reporter')
    print("Reporter matching:")
    display(summary)
    if unmatched:
        print(f"\nUnmatched reporter names ({len(unmatched)}):")
        for name in unmatched[:20]:
            print(f"  - {name}")

Reporter matching:


,metric,value
0,total_unique,203
1,matched,181
2,unmatched,22
3,match_rate,89.2%



Unmatched reporter names (22):
  - Bolivia (Plurinational State of)
  - Bosnia Herzegovina
  - Cayman Isds
  - Central African Rep.
  - China, Hong Kong SAR
  - China, Macao SAR
  - Cook Isds
  - Dem. Rep. of the Congo
  - Dominican Rep.
  - FS Micronesia
  - Faroe Isds
  - Lao People's Dem. Rep.
  - Mayotte (Overseas France)
  - Other Asia, nes
  - Rep. of Korea
  - Rep. of Moldova
  - Serbia and Montenegro (...2005)
  - Solomon Isds
  - Sudan (...2011)
  - Turks and Caicos Isds


In [14]:
if 'partner_iso3' in df_energy.columns:
    summary_p, unmatched_p = report_country_matching(df_energy, col='partner')
    print("Partner matching:")
    display(summary_p)
    if unmatched_p:
        print(f"\nUnmatched partner names ({len(unmatched_p)}):")
        for name in unmatched_p[:20]:
            print(f"  - {name}")

Partner matching:


,metric,value
0,total_unique,251
1,matched,205
2,unmatched,46
3,match_rate,81.7%



Unmatched partner names (46):
  - Areas, nes
  - Bolivia (Plurinational State of)
  - Bosnia Herzegovina
  - Br. Indian Ocean Terr.
  - Br. Virgin Isds
  - Bunkers
  - Cayman Isds
  - Central African Rep.
  - China, Hong Kong SAR
  - China, Macao SAR
  - Christmas Isds
  - Cocos Isds
  - Cook Isds
  - Dem. People's Rep. of Korea
  - Dem. Rep. of the Congo
  - Dominican Rep.
  - Europe EU, nes
  - FS Micronesia
  - Falkland Isds (Malvinas)
  - Faroe Isds


## 6. Verify Outputs

Confirm that processed files were saved correctly.

In [15]:
for f in DATA_PROCESSED.glob('*.csv'):
    size = f.stat().st_size / 1e6
    df_check = pd.read_csv(f, nrows=2)
    print(f"{f.name:30s}  {size:6.1f} MB  cols={list(df_check.columns)}")

energy_trade.csv                 128.3 MB  cols=['type_code', 'freq_code', 'ref_period_id', 'year', 'ref_month', 'reporter_code', 'reporter_iso3', 'reporter', 'flow_code', 'flow', 'partner_code', 'partner_iso3', 'partner', 'partner2code', 'partner2iso', 'partner2desc', 'classification_code', 'classification_search_code', 'is_original_classification', 'product_code', 'product', 'aggr_level', 'is_leaf', 'customs_code', 'customs_desc', 'mos_code', 'mot_code', 'mot_desc', 'qty_unit_code', 'qty_unit_abbr', 'qty', 'is_qty_estimated', 'alt_qty_unit_code', 'alt_qty_unit_abbr', 'alt_qty', 'is_alt_qty_estimated', 'net_wgt', 'is_net_wgt_estimated', 'gross_wgt', 'is_gross_wgt_estimated', 'cif_value', 'fob_value', 'trade_value_usd', 'legacy_estimation_flag', 'is_reported', 'is_aggregate', 'reporter_std', 'partner_std']
country_summary.csv                0.2 MB  cols=['country', 'year', 'total_imports', 'total_exports', 'trade_balance']
partner_summary.csv                9.8 MB  cols=['reporter', 'p

---

**Next step:** Open `02_eda.ipynb` for exploratory data analysis.